In [256]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [257]:
import numpy as np
import matplotlib.pyplot as plt
import math
import time
from src.model import FiLMResNet2In, flatten_last
from src.normalizer import RunningMeanStd
from src.envpacker import packenv, packenv_batch
from src.utils import transition_ability_batched, update_v_history
import torch
from torch import nn
import torch.nn.functional as F

## 1. Configs

In [258]:
# Training configs
AGENTS = 50     # number of agents
LEARNING_RATE = 1e-3
TRAINING_STEPS = 30000
BATCH_SIZE = 10
DISPLAY_STEP = 1000 # For visualization
TRAIN_STEP_INTERVAL = 2 # Interval of steps between training episodes


# Bewley model parameters
theta = 1 # CRRA
beta = 0.975 # Discount factor
A = 1 # Technology parameter
alpha = 0.33 # Capital share of income
gamma = 2 # Inverse Frisch elasticity
########################################### (MiLF inputs)
r = 0.04 # Interest rate (Return to savings)
w = 1 # Wage rate (Return to labor)
delta = 0.06 # Depreciation rate of capital
TAX_PARAMS = {
    "tax_consumption": 0.065,          # Consumption tax (fixed)
    "tax_income": 0.2,                # Tax on labor income
    "income_tax_elasticity": 0.5,     # Elasticity of labor supply w.r.t. after-tax income
    "saving_tax_elasticity": 0.5,     # Elasticity of savings w.r.t. after-tax income
    "tax_saving": 0.1                 # Tax on interest income
}
###########################################
p = 2.2e-6
q = 0.99

# shock parameters
# log e' = rho_v * log e + sigma_v * epsilon, epsilon ~ N(0,1)
rho_v = 0.95 # persistence of ability shock
sigma_v = 0.2 # std of ability shock
v_bar = 1.5


In [259]:
# Bounds of shock 
v_min = math.exp(-2 * sigma_v  / math.sqrt(1-rho_v**2))
v_max = math.exp( 2 * sigma_v  / math.sqrt(1-rho_v**2))

## 2. Helper functions and classes

In [260]:
def mean_across_agents(x): # Since agent number is fix, thus we can use mean instead of sum
    return torch.mean(x, dim=1, keepdim=True)


def calculate_price(a, v, h):
    a_aggregate, l_aggregate_effective = mean_across_agents(a), mean_across_agents(h*v)
    wage = A * (1-alpha) * ((a_aggregate/l_aggregate_effective) ** alpha)
    ret = A * alpha * (a_aggregate/l_aggregate_effective ** alpha)
    return wage, ret

def taxfunc(ibt, abt, taxparams=TAX_PARAMS):
    it = ibt - (1 - taxparams["tax_income"]) * (ibt**(1-taxparams["income_tax_elasticity"])/(1-taxparams["income_tax_elasticity"])) # individual after tax income
    at = abt - (1-taxparams["tax_saving"]/1-taxparams["saving_tax_elasticity"]) * (abt**(1-taxparams["saving_tax_elasticity"])) # individual after tax saving
    return it, at

def calculate_moneydisposable(wage, ret, v, h, a, delta, is_init=False):
    if is_init:
        ibt = wage * h * v   # individual before tax income
    else:
        ibt = wage * h * v + (1-delta+ret) * a   # individual before tax income

    it, at = taxfunc(ibt = ibt, abt=a)
    money_disposable = it + at

    return money_disposable, ibt


def output_transform(a, money_disposable):

    # The a here is saving rate coming from the NN output
    consumption = money_disposable * (1 - a)
    savings = money_disposable * a
    return consumption, savings


def laborfocloss(a, h, ibt, money_disposable, wage, v, taxparams=TAX_PARAMS):

    loss_foc =  -h ** (-gamma) + ((1-a)*money_disposable/(1+taxparams["tax_consumption"])) * \
        (wage * v) * (1 - taxparams["tax_income"]) * (ibt ** (-taxparams["income_tax_elasticity"]))
    
    return torch.abs(loss_foc)

# def transition_ability(
#     v_prev: torch.Tensor,
#     is_superstar_prev: torch.Tensor,
#     v_history: torch.Tensor,
#     rho_v: float,
#     sigma_v: float,
#     p: float,
#     q: float,
#     v_bar: float,
#     v_min: float,
#     v_max: float
# ) -> tuple[torch.Tensor, torch.Tensor]:
#     """
#     Calculates the ability value v_t for the next timestep based on the 
#     Bewley-Aiyagari model with a normal and a super-star state.

#     Args:
#         v_prev (torch.Tensor): Ability values from the previous timestep (v_{t-1}).
#         is_superstar_prev (torch.Tensor): Boolean tensor indicating which agents 
#                                           were in the super-star state.
#         v_history (torch.Tensor): A tensor containing the full history of 
#                                   ability values for all agents.
#         rho_v (float): Persistence parameter for the AR(1) process.
#         sigma_v (float): Volatility parameter for the AR(1) process.
#         p (float): Probability of transitioning from normal to super-star state.
#         q (float): Probability of remaining in the super-star state.
#         v_bar (float): Multiplier for super-star ability relative to the average.
#         v_min (float): Minimum bound for the ability value.
#         v_max (float): Maximum bound for the ability value.

#     Returns:
#         tuple[torch.Tensor, torch.Tensor]: A tuple containing the new ability 
#                                            values (v_t) and the new superstar status.
#     """
#     num_households = v_prev.shape[0]
    
#     # --- 1. Determine State Transitions (Logic Unchanged) ---
    
#     transitions = torch.rand(num_households, device=v_prev.device)
#     is_superstar_next = is_superstar_prev.clone()
    
#     normal_to_superstar_mask = (~is_superstar_prev) & (transitions < p)
#     is_superstar_next[normal_to_superstar_mask] = True
    
#     superstar_to_normal_mask = is_superstar_prev & (transitions >= q)
#     is_superstar_next[superstar_to_normal_mask] = False
    
#     # --- 2. Calculate Next Ability v_t ---
    
#     v_next = torch.zeros_like(v_prev)
#     normal_mask_next = ~is_superstar_next
#     superstar_mask_next = is_superstar_next
    
#     # --- For agents in the NORMAL state next period (Logic Unchanged) ---
#     if normal_mask_next.any():
#         shocks = torch.randn(normal_mask_next.sum(), device=v_prev.device)
#         log_v_next_normal = rho_v * torch.log(v_prev[normal_mask_next]) + sigma_v * shocks
#         v_next_normal = torch.exp(log_v_next_normal)
#         v_next[normal_mask_next] = torch.clamp(v_next_normal, min=v_min, max=v_max)

#     # --- For agents in the SUPER-STAR state next period (Logic Unchanged) ---
#     if superstar_mask_next.any():
#         # Calculate the historical average from the provided history tensor.
#         # Fallback to the previous period's average if history is empty (e.g., at the first step).
#         if v_history is not None and v_history.numel() > 0:
#             avg_ability = v_history.mean()
#         else:
#             avg_ability = v_prev.mean()

#         v_next[superstar_mask_next] = v_bar * avg_ability
        
#     return v_next, is_superstar_next



In [261]:
state_dim = 2*AGENTS + 2 # two state variables for each agent + 2 individual variables
cond_dim = 5 # exogenous variables for all agents in all worlds(Batch)
model = FiLMResNet2In(state_dim=state_dim, cond_dim=cond_dim,
                        hidden_dim=128, num_res_blocks=3, output_dim=3, dropout=0.1)

In [262]:
def initial_state(required_batch_size, tax_params_dict=TAX_PARAMS):
    # 隨機產生初始資產與儲蓄
    moneydisposable = np.random.uniform(0.1, 2.0, required_batch_size * AGENTS).reshape(required_batch_size, AGENTS)
    savings = np.random.uniform(0.1, 2.0, required_batch_size * AGENTS).reshape(required_batch_size, AGENTS)

    # 第一種能力 shock (v1)
    v1 = np.random.uniform(v_min, v_max, required_batch_size * AGENTS).reshape(required_batch_size, AGENTS)
    v1 = v1 / np.mean(v1, axis=1, keepdims=True)

    # 第二種能力 shock (v2)
    v2 = np.random.uniform(v_min, v_max, required_batch_size * AGENTS).reshape(required_batch_size, AGENTS)
    v2 = v2 / np.mean(v2, axis=1, keepdims=True)

    # superstar 標誌 (v1 對應一組, v2 對應一組)
    is_superstar_v1 = np.zeros((required_batch_size, AGENTS), dtype=bool)
    is_superstar_v2 = np.zeros((required_batch_size, AGENTS), dtype=bool)

    # 稅制參數轉為 tensor
    tax_params = torch.tensor(list(tax_params_dict.values()), dtype=torch.float32)
    tax_params = tax_params.repeat(required_batch_size, 1)

    # 轉為 tensor
    moneydisposable_t = torch.tensor(moneydisposable, dtype=torch.float32)
    savings_t = torch.tensor(savings, dtype=torch.float32)
    v1_t = torch.tensor(v1, dtype=torch.float32)
    v2_t = torch.tensor(v2, dtype=torch.float32)
    is_superstar_v1_t = torch.tensor(is_superstar_v1, dtype=torch.bool)
    is_superstar_v2_t = torch.tensor(is_superstar_v2, dtype=torch.bool)
    tax_params_t = tax_params

    # 回傳字典
    return {
        "moneydisposable": {"value": moneydisposable_t, "shape": tuple(moneydisposable_t.shape)},
        "savings": {"value": savings_t, "shape": tuple(savings_t.shape)},
        "v1": {"value": v1_t, "shape": tuple(v1_t.shape)},
        "v2": {"value": v2_t, "shape": tuple(v2_t.shape)},
        "is_superstar_v1": {"value": is_superstar_v1_t, "shape": tuple(is_superstar_v1_t.shape)},
        "is_superstar_v2": {"value": is_superstar_v2_t, "shape": tuple(is_superstar_v2_t.shape)},
        "tax_params": {"value": tax_params_t, "shape": tuple(tax_params_t.shape)},
    }


In [ ]:
state = initial_state(required_batch_size=256)

# print(state["moneydisposable"]["shape"])
# print(state["savings"]["shape"])
# print(state["v1"]["shape"])
# print(state["is_superstar_v1"]["shape"])
# print(state["is_superstar_v2"]["shape"])
# print(state["tax_params"]["shape"]) 


(256, 50)
(256, 50)
(256, 50)
(256, 50)
(256, 50)
(256, 5)


In [251]:
def build_inputs(moneydisposable, savings, v, is_superstar, tax_params, carry_superstar=True):
    """
    回傳:
      features : (B, A, 2A + 2)          # 給模型輸入
      condi    : (B, A, Z)              # 稅制條件
      env_info : dict                   # 僅供環境轉移使用，不進模型
    """
    B, A = moneydisposable.shape

    # (B, Z) -> (B, A, Z)
    condi = tax_params.unsqueeze(1).expand(-1, A, -1)

    # (B, 2A) -> (B, A, 2A)
    sum_info = torch.cat([moneydisposable, v], dim=1)         # (B, 2A)
    sum_info_rep = sum_info.unsqueeze(1).expand(-1, A, -1)    # (B, A, 2A)

    # (B, A, 1) × 2
    money_self = moneydisposable.unsqueeze(-1)  # (B, A, 1)
    v_self     = v.unsqueeze(-1)                # (B, A, 1)

    # 給模型的 features
    features = torch.cat([sum_info_rep, money_self, v_self], dim=2)  # (B, A, 2A+2)

    # env_info: 包含不進模型的資訊
    env_info = {}

    # 儲蓄
    env_info["savings_self"] = savings.unsqueeze(-1)  # (B, A, 1)

    # 是否 super star
    superstar = None
    if carry_superstar and is_superstar is not None:
        if is_superstar.dim() == 0:                # scalar -> (B, A, 1)
            is_superstar = is_superstar.to(moneydisposable).view(1,1).expand(B, A).unsqueeze(-1)
        elif is_superstar.dim() == 2:              # (B, A) -> (B, A, 1)
            is_superstar = is_superstar.unsqueeze(-1)
        elif is_superstar.dim() == 3:              # (B, A, 1)
            pass
        else:
            raise ValueError("is_superstar must be scalar, (B,A), or (B,A,1)")
        superstar = is_superstar.to(features.dtype)
        env_info["superstar"] = superstar

    return features, condi, env_info


In [266]:
res1 = build_inputs(
    moneydisposable=state["moneydisposable"]["value"],
    savings=state["savings"]["value"],
    v = state["v1"]["value"],
    is_superstar = state["is_superstar_v1"]["value"],
    tax_params=state["tax_params"]["value"]
)

res2 = build_inputs(
    moneydisposable=state["moneydisposable"]["value"],
    savings=state["savings"]["value"],
    v = state["v2"]["value"],
    is_superstar = state["is_superstar_v2"]["value"],
    tax_params=state["tax_params"]["value"]
)

In [ ]:
# Handle Agnets action 
# at1 (sigmoid), mut(softplus), ht (sigmoid)
acts = [torch.sigmoid, lambda x: F.softplus(x) + 1e-6, torch.sigmoid]
out = model(res2[0], res2[1])

at1, mut, ht = [acts[i](out[..., i]) for i in range(out.shape[-1])]
at1_squeezed, mut_squeezed, ht_squeezed = at1.squeeze(-1), mut.squeeze(-1), ht.squeeze(-1)

# Calculate env, and transform output 
wage, ret = calculate_price(a = state["savings"]["value"], v = state["v"]["value"], h = ht_squeezed)
money_disposable_t, ibt_t = calculate_moneydisposable(wage = wage, ret = ret, 
                                               v = state["v"]["value"], h = ht_squeezed, 
                                               a = state["savings"]["value"], delta = delta)

at1_transformed, savings = output_transform(a = at1, money_disposable = money_disposable_t)

# Shock transtion 
v_history = None

v_next, is_superstar_next = transition_ability_batched(
        v_prev=state["v"]["value"],
        is_superstar_prev=state["is_superstar"]["value"],
        v_history=v_history,            # 直接傳 tensor（或 None）
        rho_v=rho_v, sigma_v=sigma_v,
        p=p, q=q, v_bar=v_bar,
        v_min=v_min, v_max=v_max
    )

v_history = update_v_history(
    v_history=v_history,
    v_next=v_next
)


# state transition 
print(wage.shape, ret.shape, money_disposable_t.shape, ibt_t.shape)


# at1, mut = torch.sigmoid(at1), torch.exp(mut) # saving rate between 0 and 1 / ensure mu > 0
# print(res2[0].shape, res2[1].shape)
# print(model(res2[0], res2[1]).shape) # to log outputs
# print(flatten_last(model(res2[0], res2[1]))[0].shape) # to calculate loss

torch.Size([256, 1]) torch.Size([256, 1]) torch.Size([256, 50]) torch.Size([256, 50])


In [ ]:
# 